# First Iteration - 31 participants

In [26]:
import pandas as pd
import numpy as np
import os
os.chdir('/drive/notebooks/')

df = pd.read_csv("Level - Best score-Table 1.csv", sep=';', header=None, skiprows=2)
df.columns = ['Participant', 'Base Chart', 'Line chart', 'Area chart', 'Stacked area chart', 'Misleading']

def parse_pct(val):
    if pd.isna(val):
        return np.nan
    s = str(val).replace('%', '').replace(',', '.').strip()
    try:
        return float(s)
    except ValueError:
        return np.nan

levels = ['Base Chart', 'Line chart', 'Area chart', 'Stacked area chart', 'Misleading']
for col in levels:
    df[col] = df[col].apply(parse_pct)

print(f"{'Level':25s}  {'Mean':>7s}  {'SD':>7s}  {'Median':>7s}")
print("-" * 55)

for level in levels:
    vals = df[level].dropna()
    print(f"  {level:23s}  {vals.mean():6.1f}%  {vals.std():6.1f}%  {vals.median():6.0f}%")

Level                         Mean       SD   Median
-------------------------------------------------------
  Base Chart                 85.5%    18.0%      90%
  Line chart                 92.1%     8.3%      93%
  Area chart                 94.3%    23.2%     100%
  Stacked area chart         78.0%    23.2%      84%
  Misleading                 92.0%    12.9%     100%


In [23]:

import os
os.chdir('/drive/notebooks/')

import glob
import re
from collections import defaultdict

log_files = sorted(glob.glob("activityLog*.txt"))

def normalize_level(name):
    s = name.lower().strip()
    if 'misleading' in s:
        return 'Misleading'
    elif 'stacked' in s:
        return 'Stacked Area'
    elif 'area' in s:
        return 'Area Chart'
    elif 'line' in s:
        return 'Line Chart'
    elif 'base' in s:
        return 'Base Chart'
    return name

LEVELS = ['Base Chart', 'Line Chart', 'Area Chart', 'Stacked Area', 'Misleading']

all_actions = []

for filepath in log_files:
    filename = filepath.split('/')[-1].split('\\')[-1]
    current_level = None
    current_activity = None

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            level_match = re.search(r'=== LEVEL (.+?) START', line)
            if level_match:
                current_level = normalize_level(level_match.group(1))
                continue

            activity_match = re.search(r'=== START ACTIVITY: (.+?) ===', line)
            if activity_match:
                current_activity = activity_match.group(1)
                continue

            if '=== LEVEL END' in line:
                current_level = None
                current_activity = None
                continue

            if current_level and '| ACTION:' in line:
                ts_match = re.match(r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})', line)
                if not ts_match:
                    continue

                is_error = '❌' in line
                is_correct = '✅' in line
                if not is_error and not is_correct:
                    continue

                all_actions.append({
                    'timestamp_str': ts_match.group(1),
                    'correct': is_correct,
                    'level': current_level,
                    'activity': current_activity,
                    'file': filename,
                })

all_actions.sort(key=lambda a: (a['file'], a['timestamp_str']))
deduped = []
prev_key = None
for a in all_actions:
    key = (a['timestamp_str'], a['correct'], a['level'])
    if key == prev_key:
        continue
    prev_key = key
    deduped.append(a)

errors = [a for a in deduped if not a['correct']]

errors_by_level = defaultdict(int)
errors_by_activity = defaultdict(lambda: defaultdict(int))
errors_per_file = defaultdict(int)

for e in errors:
    errors_by_level[e['level']] += 1
    errors_by_activity[e['level']][e['activity']] += 1
    errors_per_file[e['file']] += 1

total = sum(errors_by_level.values())

print(f"{'Level':25s}  {'Errors':>7s}")
print("-" * 35)
for level in LEVELS:
    print(f"  {level:23s}  {errors_by_level[level]:6d}")
print("-" * 35)
print(f"  {'TOTAL':23s}  {total:6d}")

print(f"\n\nERRORS BY ACTIVITY WITHIN EACH LEVEL")
print("=" * 60)
for level in LEVELS:
    activities = errors_by_activity[level]
    if activities:
        print(f"\n  {level}:")
        for act, count in sorted(activities.items(), key=lambda x: -x[1]):
            print(f"    {act:45s}  {count:4d}")


Level                       Errors
-----------------------------------
  Base Chart                  234
  Line Chart                  266
  Area Chart                   94
  Stacked Area                849
  Misleading                  153
-----------------------------------
  TOTAL                      1596


ERRORS BY ACTIVITY WITHIN EACH LEVEL

  Base Chart:
    Axes Activity                                   234

  Line Chart:
    Prepare Second Category                         105
    Second Line Drawing Activity                     60
    Point Plotting Activity                          59
    Second Point Plotting Activity                   34
    Line Drawing Activity                             8

  Area Chart:
    Area Comparison Activity                         94

  Stacked Area:
    First Stacking Activity                         311
    Second Point Plotting Activity                  192
    Point Plotting Activity                         181
    Second Line Drawing Acti